In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy.ndimage import gaussian_filter
import ot 

In [ ]:
df_meme_agg_folder = "/scratch_share/mind/d.pizzo-thesis/DF_MEME_AGGREGATED"

#saliency_last = "/scratch_share/mind/d.pizzo-thesis/SALIENCY_MAPS_MBLIP_LAST[-2]"
#saliency_full = "/scratch_share/mind/d.pizzo-thesis/SALIENCY_MAPS_MBLIP_FULL[-39]"

#output_last = "/scratch_share/mind/d.pizzo-thesis/metrics_mblip_last[-2].csv"
#output_full = "/scratch_share/mind/d.pizzo-thesis/metrics_mblip_full[-2].csv"

#df_mclip_last = pd.read_csv("/scratch_share/mind/d.pizzo-thesis/mclip_last.csv")
df_mclip_full = pd.read_csv("/scratch_share/mind/d.pizzo-thesis/mclip_full.csv")

#output_graphs = "/scratch_share/mind/d.pizzo-thesis/plots/"
#os.makedirs(output_graphs, exist_ok=True)

In [ ]:
def set_values():

    screen_width, screen_height = 1920, 1080
    meme_width, meme_height = 768, 768
    offset_x = (screen_width - meme_width) // 2
    offset_y = (screen_height - meme_height) // 2
    heatmap = np.zeros((meme_height, meme_width), dtype=float)
    return offset_x, offset_y, heatmap

def set_heatmap(df, offset_x, offset_y, base_heatmap):

    heatmap = base_heatmap.copy()
    for _, row in df.iterrows():
        x = int(round(row['CURRENT_FIX_X'] - offset_x))
        y = int(round(row['CURRENT_FIX_Y'] - offset_y))
        duration = row['CURRENT_FIX_DURATION']
        if 0 <= x < 768 and 0 <= y < 768:
            heatmap[y, x] += duration
    heatmap = gaussian_filter(heatmap, sigma=20)
    return heatmap

In [ ]:
def NSS(pred_map, gt_fix):
    pred_z = (pred_map - np.mean(pred_map)) / (np.std(pred_map) + 1e-8)
    return np.mean(pred_z[gt_fix > 0])

def EMD_2d(pred_map, gt_map, size=64):
    cam_small    = cv2.resize(pred_map, (size, size))
    gt_map_small = cv2.resize(gt_map,   (size, size))

    a = cam_small.ravel().astype(np.float64)
    b = gt_map_small.ravel().astype(np.float64)
    a /= (a.sum() + 1e-8)
    b /= (b.sum() + 1e-8)

    x, y = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    coords = np.stack([x.ravel(), y.ravel()], axis=1)
    M = ot.dist(coords, coords, metric="euclidean")

    return ot.emd2(a, b, M)


In [ ]:
def evaluate_model(saliency_folder, output_csv):
    """
    per ogni meme
    1. Carica heatmap umana aggregata da DF_MEME_AGGREGATED
    2. Carica heatmap modello dal file .npy
    3. Calcola NSS e EMD
    4. Salva risultati in CSV
    """

    offset_x, offset_y, base_heatmap = set_values()
    results = []

    # Lista meme dai file .npy disponibili
    npy_files = [f for f in os.listdir(saliency_folder) if f.endswith('.npy')]
    print(f"Trovati {len(npy_files)} file .npy in {saliency_folder}")

    for npy_file in sorted(npy_files):
        # Ricava il nome del meme dal nome del file
        # es. "meme_1011_gradcam_and_ig.npy" → "meme_1011.jpg"
        meme_name = npy_file.replace("_gradcam_and_ig.npy", ".jpg")
        print(f"[INFO] Elaboro: {meme_name}")

        # heatmap umana
        agg_path = os.path.join(df_meme_agg_folder, f"{meme_name}.csv")
        if not os.path.exists(agg_path):
            print(f"[WARN] File aggregato non trovato: {agg_path} — skip")
            continue

        meme_aggregated = pd.read_csv(agg_path)
        human_heatmap   = set_heatmap(meme_aggregated, offset_x, offset_y, base_heatmap)

        # ground truth per NSS
        gt_fix = np.zeros((768, 768))
        for _, row in meme_aggregated.iterrows():
            x = int(round(row['CURRENT_FIX_X'] - offset_x))
            y = int(round(row['CURRENT_FIX_Y'] - offset_y))
            if 0 <= x < 768 and 0 <= y < 768:
                gt_fix[y, x] = 1

        #ground truth per EMD 
        gt_map = human_heatmap.copy()
        gt_map = (gt_map - gt_map.min()) / (gt_map.max() + 1e-8)

        # heatmap modello
        npy_path = os.path.join(saliency_folder, npy_file)
        cam = np.load(npy_path)

        # metriche
        nss = NSS(cam, gt_fix)
        emd = EMD_2d(cam, gt_map)

        results.append({
            'meme_name': meme_name,
            'NSS': nss,
            'EMD': emd
        })

        print(f"  NSS: {nss:.4f} | EMD: {emd:.4f}")

    # Salva CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"\n[OK] Salvato: {output_csv}")
    print(f"NSS medio: {df_results['NSS'].mean():.4f}")
    print(f"EMD medio: {df_results['EMD'].mean():.4f}")

    return df_results

### Lastlayer fine tuned


In [ ]:

print("Evaluation mBLIP Last Layer Fine-Tuned")
df_last = evaluate_model(saliency_last, output_last)

### BOXPLOT GENERALE CON TUTTI LAYER LAST LAYER FINE TUNED

In [ ]:

#config
base = "/scratch_share/mind/d.pizzo-thesis"          # cartella dei CSV
output_dir = "/scratch_share/mind/d.pizzo-thesis/plots"
os.makedirs(output_dir, exist_ok=True)

layers = list(range(-1, -40, -1))   # [-1, -2, ..., -39]

def csv_path(layer):
    # il layer -1 è salvato come "metrics_mblip_last"
    if layer == -1:
        return os.path.join(base, "metrics_mblip_last.csv")
    return os.path.join(base, f"metrics_mblip_last[{layer}].csv")

#CARICA E CONCATENA TUTTI I CSV
dfs = []
for layer in layers:
    path = csv_path(layer)
    if not os.path.exists(path):
        print(f"[WARN] File mancante per layer {layer}: {path} — skip")
        continue
    df = pd.read_csv(path)
    df['layer'] = layer
    dfs.append(df)

df_all = pd.concat(dfs).reset_index(drop=True)
print(f"Caricati {df_all['layer'].nunique()} layer, {len(df_all)} righe totali")

#asse x: dal primo layer della rete (-39) all'ultimo (-1)
order = sorted(df_all['layer'].unique())   # [-39, -38, ..., -1]

# BOXPLOT
for col, color in [('NSS', 'plum'), ('EMD', 'gold')]:
    plt.figure(figsize=(20, 6))
    ax = sns.boxplot(data=df_all, x='layer', y=col,
                     order=order, color=color, fliersize=2)

    # medie per layer (allineate all'ordine dell'asse x)
    means = df_all.groupby('layer')[col].mean().reindex(order)

    # linea per mostrare l'andamento della media nel grafico
    plt.plot(range(len(order)), means.values,
             color='black', marker='o', markersize=4,
             linewidth=1.5, label='Media')

    # scrittura valore medio 
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin
    ax.set_ylim(ymin, ymax + span * 0.15)      # spazio in alto per le etichette
    y_text = ymax + span * 0.02
    for i, m in enumerate(means.values):
        ax.text(i, y_text, f'{m:.2f}',
                ha='center', va='bottom', rotation=90,
                fontsize=8, color='black')

    plt.title(f"{col} per layer — mBLIP Last Layer Fine-Tuned",
              fontsize=16, weight='bold')
    plt.xlabel("Layer (da -39 = primo a -1 = ultimo)", fontsize=13)
    plt.ylabel(col, fontsize=13)
    plt.xticks(rotation=90, fontsize=9)
    plt.yticks(fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"{col}_all_layers_mblip_last.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Salvato: {filename}")

# TABELLA RIASSUNTIVA 
summary = df_all.groupby('layer')[['NSS', 'EMD']].mean().reindex(order).round(3)
print("\nMedia NSS ed EMD per layer:")
print(summary)

### full fine

In [ ]:
print("Evaluation mBLIP Full Fine-Tuned")
df_full = evaluate_model(saliency_full, output_full)

### Boxplot generale per tutti i layer full fine tuned


In [ ]:
# config
base = "/scratch_share/mind/d.pizzo-thesis"         
output_dir = "/scratch_share/mind/d.pizzo-thesis/plots"
os.makedirs(output_dir, exist_ok=True)

layers = list(range(-1, -40, -1))   # [-1, -2, ..., -39]

def csv_path(layer):
    # il layer -1 è salvato come "metrics_mblip_last"
    if layer == -1:
        return os.path.join(base, "metrics_mblip_full.csv")
    return os.path.join(base, f"metrics_mblip_full[{layer}].csv")

#CARICA E CONCATENA TUTTI I CSV
dfs = []
for layer in layers:
    path = csv_path(layer)
    if not os.path.exists(path):
        print(f"[WARN] File mancante per layer {layer}: {path} — skip")
        continue
    df = pd.read_csv(path)
    df['layer'] = layer
    dfs.append(df)

df_all = pd.concat(dfs).reset_index(drop=True)
print(f"Caricati {df_all['layer'].nunique()} layer, {len(df_all)} righe totali")

# asse x: dal primo layer della rete (-39) all'ultimo (-1)
order = sorted(df_all['layer'].unique())   # [-39, -38, ..., -1]

# BOXPLOT 
for col, color in [('NSS', 'plum'), ('EMD', 'gold')]:
    plt.figure(figsize=(20, 6))
    ax = sns.boxplot(data=df_all, x='layer', y=col,
                     order=order, color=color, fliersize=2)

    # medie per layer (allineate all'ordine dell'asse x)
    means = df_all.groupby('layer')[col].mean().reindex(order)

    #linea che mostra andamento media
    plt.plot(range(len(order)), means.values,
             color='black', marker='o', markersize=4,
             linewidth=1.5, label='Media')

    # scrittura valore medio su grafico
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin
    ax.set_ylim(ymin, ymax + span * 0.15)      # spazio in alto per le etichette
    y_text = ymax + span * 0.02
    for i, m in enumerate(means.values):
        ax.text(i, y_text, f'{m:.2f}',
                ha='center', va='bottom', rotation=90,
                fontsize=8, color='black')

    plt.title(f"{col} per layer — mBLIP Full-Fine Tuned",
              fontsize=16, weight='bold')
    plt.xlabel("Layer (da -39 = primo a -1 = ultimo)", fontsize=13)
    plt.ylabel(col, fontsize=13)
    plt.xticks(rotation=90, fontsize=9)
    plt.yticks(fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"{col}_all_layers_mblip_full.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Salvato: {filename}")

# TABELLA RIASSUNTIVA 
summary = df_all.groupby('layer')[['NSS', 'EMD']].mean().reindex(order).round(3)
print("\nMedia NSS ed EMD per layer:")
print(summary)

In [ ]:
print("CONFRONTO TRA I DUE MODELLI")

for metric in ['NSS', 'EMD']:
    print(f"\n--- {metric} ---")
    print("Last Layer Fine-Tuned:")
    print(df_last[metric].describe())
    print("\nFULL Fine-Tuned:")
    print(df_full[metric].describe())

### Graphs


In [ ]:
df_last = pd.read_csv(output_last)
df_full = pd.read_csv(output_full)

df_last['Model'] = 'Last Layer Fine-Tuned'
df_full['Model'] = 'Full-Fine-Tuned'
df_all = pd.concat([df_last, df_full])

metrics = ['NSS', 'EMD']
colors  = {'Last Layer Fine-Tuned': 'plum', 'Full-Fine-Tuned': 'gold'}

In [ ]:
for col in metrics:
    for df, name, color in [
        (df_last, 'Last Layer Fine-Tuned', 'plum'),
        (df_full, 'Full-Fine-Tuned',   'gold')
    ]:
        plt.figure(figsize=(8, 5))
        plt.hist(df[col], bins=30, color=color,
                 edgecolor='black', alpha=0.7)
        plt.title(f"Distribution of {col} — {name}",
                  fontsize=16, weight='bold')
        plt.xlabel(col, fontsize=14)
        plt.ylabel("Frequency", fontsize=14)
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.tight_layout()

        filename = os.path.join(
            output_graphs,
            f"{col}_{name.replace(' ', '_')}_hist.png"
        )
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Salvato: {filename}")

In [ ]:
for col in metrics:
    plt.figure(figsize=(8, 5))
    plt.hist(df_last[col], bins=30, color='plum',
             edgecolor='black', alpha=0.6,
             label='Last Layer Fine-Tuned')
    plt.hist(df_full[col], bins=30, color='gold',
             edgecolor='black', alpha=0.6,
             label='Q-Former Fine-Tuned')
    plt.title(f"Comparison of {col} — mBLIP",
              fontsize=16, weight='bold')
    plt.xlabel(col, fontsize=14)
    plt.ylabel("Frequency", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()

    filename = os.path.join(output_graphs, f"{col}_comparison_hist.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Salvato: {filename}")

In [ ]:
for col in metrics:
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=df_all, x='Model', y=col,
        hue='Model',
        palette=['plum', 'gold']
    )
    plt.title(f"Distribution of {col} — Both Models",
              fontsize=16, weight='bold')
    plt.ylabel(col, fontsize=14)
    plt.xlabel("Model", fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()

    filename = os.path.join(output_graphs, f"{col}_boxplot_comparison.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Salvato: {filename}")

In [ ]:
for col in metrics:
    print(f"\n{'='*50}")
    print(f"TOP 10 e BOTTOM 10 — {col}")
    print(f"{'='*50}")

    for df, name in [
        (df_last, 'Last Layer Fine-Tuned'),
        (df_full, 'Q-Former Fine-Tuned')
    ]:
        print(f"\n--- {name} ---")
        print(f"Top 10 {col}:")
        print(df.nlargest(10, col)[['meme_name', col]].to_string(index=False))
        print(f"\nBottom 10 {col}:")
        print(df.nsmallest(10, col)[['meme_name', col]].to_string(index=False))

print("\nTutti i grafici salvati!")

In [ ]:
df_last = pd.read_csv(output_last)
df_full = pd.read_csv(output_full)
df_last['Model'] = 'mBLIP Fixed Features'
df_full['Model'] = 'mBLIP Full Finetune'
df_mclip_last['Model'] = 'mCLIP Fixed Features'
df_mclip_full['Model'] = 'mCLIP Full Finetune'

In [ ]:
cols = ['NSS', 'EMD', 'Model']
df_all = pd.concat([
    df_last[cols],
    df_full[cols],
    df_mclip_last[cols],
    df_mclip_full[cols]
]).reset_index(drop=True)

In [ ]:
order = ['mBLIP Fixed Features', 'mBLIP Full Finetune',
         'mCLIP Fixed Features', 'mCLIP Full Finetune']
colors = ['plum', 'gold', 'skyblue', 'lightgreen']

In [ ]:
#boxplot di confronto
for col in ['NSS', 'EMD']:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_all, x='Model', y=col,
                order=order, hue='Model', palette=colors, legend=False)

    # media
    means = df_all.groupby('Model')[col].mean()
    for i, model in enumerate(order):
        plt.text(i, means[model], f'{means[model]:.2f}',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)

    plt.title(f"Confronto {col} — mBLIP vs mCLIP", fontsize=16, weight='bold')
    plt.ylabel(col, fontsize=14)
    plt.xlabel("")
    plt.xticks(rotation=15, fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    filename = os.path.join(output_graphs, f"{col}_confronto_4_modelli.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Salvato: {filename}")

In [ ]:
print("RIASSUNTO MEDIE")
for col in ['NSS', 'EMD']:
    print(f"\n--- {col} ---")
    print(df_all.groupby('Model')[col].agg(['mean', 'std', 'min', 'max']).round(3))

## EVALUATION LAYERS X HEAD

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model_tag = "full"
base = "/scratch_share/mind/d.pizzo-thesis/head_attention_mblip"
output_dir = os.path.join(base, "matrici")
os.makedirs(output_dir, exist_ok=True)

layers = list(range(-1, -40, -1))   # [-1, -2, ..., -39]

# unisco tutti i 39 csv
dfs = []
for layer in layers:
    path = os.path.join(base, f"metrics_heads_mblip_{model_tag}[{layer}].csv")
    if not os.path.exists(path):
        print(f"File mancante: {path}")
        continue
    df = pd.read_csv(path)
    df['layer'] = layer
    dfs.append(df)

df_all = pd.concat(dfs).reset_index(drop=True)
print(f"Caricati {df_all['layer'].nunique()} layer, {len(df_all)} righe totali")


# matrice layer x heaf
for metric, cmap in [('NSS', 'YlGnBu'), ('EMD', 'YlOrRd_r')]:

    matrix = df_all.pivot_table(index='layer', columns='head',
                                values=metric, aggfunc='mean')
    matrix = matrix.reindex(index=layers)   # righe da -1 (in alto) a -39 (in basso)

    # Etichette: layer come ordinale 1°-39°, head da 1 a 16
    ylabels = [f"{40 - abs(l)}°" for l in matrix.index]   # -1 -> 39°, -39 -> 1°
    xlabels = [str(h + 1) for h in matrix.columns]

    plt.figure(figsize=(14, 20))
    sns.heatmap(matrix, annot=True, fmt=".2f",
                cmap=cmap, linewidths=0.5, linecolor='white',
                annot_kws={"fontsize": 6},
                xticklabels=xlabels, yticklabels=ylabels,
                cbar_kws={'label': f'{metric} medio'})

    plt.title(f"{metric} medio Layer × Head — mBLIP ({model_tag})",
              fontsize=16, weight='bold')
    plt.xlabel("Head", fontsize=13)
    plt.ylabel("Layer", fontsize=13)
    plt.yticks(rotation=0)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"matrix_{metric}_mblip_{model_tag}.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Salvato: {filename}")

    # Combinazione migliore
    if metric == 'NSS':
        best = matrix.stack().idxmax();  val = matrix.stack().max()
    else:
        best = matrix.stack().idxmin();  val = matrix.stack().min()
    print(f"  Migliore {metric}: layer {best[0]} ({40 - abs(best[0])}°), "
          f"head {best[1] + 1} → {val:.3f}\n")

In [ ]:

model_tag = "fine"
base = "/scratch_share/mind/d.pizzo-thesis/head_attention_mblip"
output_dir = os.path.join(base, "matrici")
os.makedirs(output_dir, exist_ok=True)

layers = list(range(-1, -40, -1))   # [-1, -2, ..., -39]

# unisco tutti i 39 csv
dfs = []
for layer in layers:
    path = os.path.join(base, f"metrics_heads_mblip_{model_tag}[{layer}].csv")
    if not os.path.exists(path):
        print(f" File mancante: {path} ")
        continue
    df = pd.read_csv(path)
    df['layer'] = layer
    dfs.append(df)

df_all = pd.concat(dfs).reset_index(drop=True)
print(f"Caricati {df_all['layer'].nunique()} layer, {len(df_all)} righe totali")


# matrice layer x heaf
for metric, cmap in [('NSS', 'YlGnBu'), ('EMD', 'YlOrRd_r')]:

    matrix = df_all.pivot_table(index='layer', columns='head',
                                values=metric, aggfunc='mean')
    matrix = matrix.reindex(index=layers)   # righe da -1 (in alto) a -39 (in basso)

    # Etichette: layer come ordinale 1°-39°, head da 1 a 16
    ylabels = [f"{40 - abs(l)}°" for l in matrix.index]   # -1 -> 39°, -39 -> 1°
    xlabels = [str(h + 1) for h in matrix.columns]

    plt.figure(figsize=(14, 20))
    sns.heatmap(matrix, annot=True, fmt=".2f",
                cmap=cmap, linewidths=0.5, linecolor='white',
                annot_kws={"fontsize": 6},
                xticklabels=xlabels, yticklabels=ylabels,
                cbar_kws={'label': f'{metric} medio'})

    plt.title(f"{metric} medio Layer × Head — mBLIP ({model_tag})",
              fontsize=16, weight='bold')
    plt.xlabel("Head", fontsize=13)
    plt.ylabel("Layer", fontsize=13)
    plt.yticks(rotation=0)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"matrix_{metric}_mblip_{model_tag}.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Salvato: {filename}")

    # Combinazione migliore
    if metric == 'NSS':
        best = matrix.stack().idxmax();  val = matrix.stack().max()
    else:
        best = matrix.stack().idxmin();  val = matrix.stack().min()
    print(f"  Migliore {metric}: layer {best[0]} ({40 - abs(best[0])}°), "
          f"head {best[1] + 1} → {val:.3f}\n")